In [ ]:
## resampled_bms data(매트랩 코드에서 보간 후) current > 0 행만 자르기

import pandas as pd
from pathlib import Path
from typing import Optional

def trim_bms(
    input_path: str,
    output_path: Optional[str] = None,   # None → 원본 덮어쓰기
):
    """
    1) Current > 0 이 처음 등장한 행(idx_curr) 찾기
    2) idx_curr 의 바로 한 행 앞(start_row)부터 끝까지 남김
    3) [Current, Temp1, Temp2, Temp4] 열만 추출해 이름 변경
       ─ Current  → BMS current
       ─ Temp1    → Busbar Temp 1
       ─ Temp2    → Busbar Temp 2
       ─ Temp4    → Busbar Temp 3
    4) CSV / Excel 저장 (output_path 없으면 원본 덮어쓰기)
    """

    # ── 0. 파일 읽기 ─────────────────────────────────────────────
    p_in = Path(input_path)
    if p_in.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(p_in)
    else:
        df = pd.read_csv(p_in, encoding="euc-kr")
    
    # ── 1. Current 열을 float 로만 변환 (단위 'A' 처리 없음) ───────
    df["Current"] = df["Current"].astype(float)

    # Current > 0 첫 위치 찾기
    mask = df["Current"] != 0
    if not mask.any():
        raise ValueError("Current != 0 인 행이 없습니다.")
    idx_curr  = mask.idxmax()
    start_row = max(idx_curr - 1, 0)     # 한 행 앞 (0 미만 방지)

    # ── 2. 슬라이스 & 열 이름 변경 ───────────────────────────────
    trimmed = (
        df.loc[start_row:, ["Current", "Temp1", "Temp2", "Temp4"]]
          .rename(columns={
              "Current": "BMS current",
              "Temp1"  : "Busbar Temp 1",
              "Temp2"  : "Busbar Temp 2",
              "Temp4"  : "Busbar Temp 3",
          })
          .reset_index(drop=True)
    )

    # ── 3. 저장 (output_path 없으면 원본 덮어쓰기) ────────────────
    if output_path is None:
        output_path = p_in.with_stem(p_in.stem + "_trimmed").with_suffix(".xlsx")
    p_out = Path(output_path)
    p_out.parent.mkdir(parents=True, exist_ok=True)

    if p_out.suffix.lower() == ".csv":
        trimmed.to_csv(p_out, index=False)
    else:
        if p_out.suffix == "":
            p_out = p_out.with_suffix(".xlsx")
        trimmed.to_excel(p_out, index=False)

    print(f"✅ Saved trimmed log → {p_out}")

# 사용 예시 (원본 파일 덮어쓰기)
trim_bms(
    input_path=r"test data/6060_regen70 데이터/resampled_6060bms.xlsx"
)


In [ ]:
## 사이클러 데이터 trim (soc 설정 이후)
import pandas as pd
from pathlib import Path
from typing import Union, Optional
import re

def trim_before_first_nonzero_remaining(
    input_path : Union[str, Path],
    output_path: Optional[Union[str, Path]] = None,
    zero_tol   : float = 0.0,          # |val| ≤ zero_tol → 0 으로 간주
    sheet_name : str   = "record"      # Excel 시트 이름
):
    """
    남은 용량(Remaining Capa*) 값이 처음 0이 아닌 행의 '직전 행'부터 유지해 저장.
    """

    p_in = Path(input_path)

    # ── 1. 파일 읽기 ─────────────────────────────────────────
    try:
        df = pd.read_excel(p_in, sheet_name=sheet_name)
    except ValueError:                        # 지정 시트 없으면 첫 시트
        df = pd.read_excel(p_in)
        sheet_name = "Sheet1"
    df.rename(columns={"SOC/DOD(%)": "SOC(%)"}, inplace=True)
    
    # ── 2. Remaining Capa 열 탐색  (띄어쓰기·대/소문자 무시) ──
    def _norm(s: str) -> str:
        return re.sub(r"\s+", "", s).lower()      # 공백 제거 후 소문자

    cand = [
        c for c in df.columns
        if "remaining" in _norm(c) and "capa" in _norm(c)
    ]
    if not cand:
        raise ValueError("'Remaining Capa' 열을 찾지 못했습니다.")
    col_rem = cand[0]         # 첫 번째 매칭 열 사용

    # ── 3. 첫 0 아닌 행 인덱스 ────────────────────────────────
    df[col_rem] = pd.to_numeric(df[col_rem], errors="coerce")
    nz_mask = df[col_rem].abs() > zero_tol
    if not nz_mask.any():
        raise ValueError("Remaining Capa 가 0 이 아닌 행이 없습니다.")

    idx_first = nz_mask.idxmax()           # 첫 True
    start_row = max(idx_first - 1, 0)      # 그 직전 행부터
    trimmed   = df.loc[start_row:].reset_index(drop=True)

    # ── 4. 저장 경로 ─────────────────────────────────────────
    if output_path is None:
        output_path = p_in.with_stem(p_in.stem + "_trimmed").with_suffix(".xlsx")
    p_out = Path(output_path)
    p_out.parent.mkdir(parents=True, exist_ok=True)

        # ── 5. 저장  (시트 구조 유지) ────────────────────────────
    if p_out.suffix.lower() == ".csv":
        trimmed.to_csv(p_out, index=False, float_format="%.4f")  # ← 여기에!
    else:
        trimmed.to_excel(p_out, index=False, sheet_name=sheet_name)

    print(f"첫 Remaining Capa ≠ 0 행 = {idx_first}  →  "
          f"{start_row} 행부터 남겨 저장: {p_out}")

trim_before_first_nonzero_remaining(
    r"Dtest data/6060_regen70 데이터/resampled_6060cycler.xlsx",
    # output_path=None,        # 생략 시 *_trimmed.xlsx 로 저장
    zero_tol   = 0.0           # 필요 시 0.001 등 조정
)


In [ ]:
## 최종 합치기

import pandas as pd
from pathlib import Path

# ── 경로 설정 ─────────────────────────────────────────────
cycler_path = Path(r"test data/6060_regen70 데이터/resampled_6060cycler_trimmed.xlsx")
bms_path    = Path(r"test data/6060_regen70 데이터/resampled_6060bms_trimmed.xlsx")
tctemp_path = Path(r"test data/6060_regen70 데이터/resampled_6060tctemp.xlsx")
out_path    = Path(r"test data/6060_regen70 데이터/test data_6060_regen70.xlsx")

# ── 파일 읽기 ─────────────────────────────────────────────
df_cycler  = pd.read_excel(cycler_path).reset_index(drop=True)
df_bms     = pd.read_excel(bms_path).iloc[:, 1:].reset_index(drop=True)     # 첫 열 제외
df_tctemp  = pd.read_excel(tctemp_path).iloc[:, 1:].reset_index(drop=True)  # 첫 열 제외

# ── 가장 긴 길이에 맞춰 인덱스 확장(짧은 쪽은 NaN) ─────────────────────
max_len = max(len(df_cycler), len(df_bms), len(df_tctemp))

dfs = [df.reindex(range(max_len)) for df in (df_cycler, df_bms, df_tctemp)]

# ── 단순 이어붙이기 (cycler → bms → tctemp) ───────────────────────────
df_merged = pd.concat(dfs, axis=1)

# ── 저장 ─────────────────────────────────────────────────
df_merged.to_excel(out_path, index=False)
print(f"병합 완료 → {out_path}")

In [ ]:
## 안씀
# tctemp 첫 행 처리
# trim_temp_rise.py
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def trim_to_temp_rise(
    input_path: str,
    time_col: str = "Time_s",   # A열
    temp_col: str = "TCTemp3",  # B열
    baseline_rows: int = 10,    # 기준선 계산용 구간(초기 n행 평균)
    rise_threshold: float = 0.5 # 기준선 대비 몇 °C 올라가면 “상승 시작”으로 볼지
):
    """
    ▸ 온도가 rise_threshold °C 이상 올라간 첫 번째 행을 기준으로 앞부분 제거
    ▸ 잘린 결과를 (원본파일명_trimmed.xlsx , _trimmed.png) 형태로 저장
    """
    p = Path(input_path).expanduser().resolve()
    df = pd.read_excel(p)

    # ① 기준선(초기 n행 평균) 계산
    baseline = df.loc[:baseline_rows - 1, temp_col].mean()

    # ② 상승 시작 행 인덱스 탐색
    try:
        first_rise_idx = df.index[df[temp_col] > baseline + rise_threshold][0]
    except IndexError:
        raise ValueError("조건을 만족하는 상승 구간을 찾지 못했습니다. "
                         "rise_threshold를 낮춰 보세요.")

    # ③ 앞부분 절단 & 인덱스 재설정
    trimmed = df.loc[first_rise_idx:].reset_index(drop=True)

    # ④ 결과 저장 경로 구성
    out_base = p.with_name(p.stem + "_trimmed")
    xlsx_path = out_base.with_suffix(".xlsx")
    png_path  = out_base.with_suffix(".png")

    # ⑤ Excel 저장
    trimmed.to_excel(xlsx_path, index=False)

    # ⑥ 그래프 저장
    plt.figure(figsize=(10, 5))
    plt.plot(trimmed[time_col], trimmed[temp_col])
    plt.xlabel("Time (s)")
    plt.ylabel("Temperature (°C)")
    plt.title("Temperature after Rise")
    plt.tight_layout()
    plt.savefig(png_path, dpi=300)
    plt.close()

    print(f"Trimmed data  : {xlsx_path}")
    print(f"Plot saved    : {png_path}")

if __name__ == "__main__":
    trim_to_temp_rise("D:/race/기술 아이디어/test data/7060_regen70 데이터/resampled_7060tctemp.xlsx")  # 필요하면 경로 수정
